In [1]:
from pyspark import SparkContext, SparkConf
import datetime

BASE_PATH = "/Lab03_DS/"

## Bài 1: Tính điểm trung bình và tổng số lượt đánh giá cho mỗi phim

In [2]:
movies_rdd = sc.textFile(BASE_PATH + "movies.txt").map(lambda line: line.split(","))
movie_map = movies_rdd.map(lambda x: (x[0], x[1])).collectAsMap()

ratings_rdd = sc.textFile(BASE_PATH + "ratings_1.txt," + BASE_PATH + "ratings_2.txt").map(lambda line: line.split(","))
movie_rating_rdd = ratings_rdd.map(lambda x: (x[1], (float(x[2]), 1)))

movie_total_rdd = movie_rating_rdd.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
movie_avg_rdd_5 = movie_total_rdd.filter(lambda x: x[1][1] >= 5).mapValues(lambda x: x[0] / x[1])
movie_avg_rdd_50 = movie_total_rdd.filter(lambda x: x[1][1] >= 50).mapValues(lambda x: x[0] / x[1])

if not movie_avg_rdd_5.isEmpty():
    highest_rated_5 = movie_avg_rdd_5.sortBy(lambda x: x[1], ascending=False).first()
    movie_id, avg_rating = highest_rated_5
    title = movie_map.get(movie_id, "Unknown")
    print(f"Phim có điểm cao nhất(5 lượt đánh giá): {title} (ID: {movie_id}) với điểm trung bình {avg_rating}")
else:
    print("Không có phim nào đạt đủ 5 lượt đánh giá.")

if not movie_avg_rdd_50.isEmpty():
    highest_rated_50 = movie_avg_rdd_50.sortBy(lambda x: x[1], ascending=False).first()
    movie_id, avg_rating = highest_rated_50
    title = movie_map.get(movie_id, "Unknown")
    print(f"Phim có điểm cao nhất(50 lượt đánh giá): {title} (ID: {movie_id}) với điểm trung bình {avg_rating}")
else:
    print("Không có phim nào đạt đủ 50 lượt đánh giá.")

Phim có điểm cao nhất(5 lượt đánh giá): Sunset Boulevard (1950) (ID: 1015) với điểm trung bình 4.357142857142857
Không có phim nào đạt đủ 50 lượt đánh giá.


## Bài 2: Phân tích đánh giá theo thể loại

In [3]:
movie_genres_rdd = movies_rdd.map(lambda x: (x[0], x[2].split("|")))
movie_rating_b2_rdd = ratings_rdd.map(lambda x: (x[1], float(x[2])))

joined_b2_rdd = movie_rating_b2_rdd.join(movie_genres_rdd)

def flat_map_genres(x):
    movie_id, (rating, genres) = x
    return [(genre, (rating, 1)) for genre in genres]

genre_rating_rdd = joined_b2_rdd.flatMap(flat_map_genres)
genre_total_rdd = genre_rating_rdd.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
genre_avg_rdd = genre_total_rdd.mapValues(lambda x: x[0] / x[1])

print("Điểm trung bình theo thể loại (10 kết quả đầu):")
for genre, avg_rating in genre_avg_rdd.take(10):
    print(f"Thể loại: {genre} - Điểm trung bình: {avg_rating:.2f}")

Điểm trung bình theo thể loại (10 kết quả đầu):


Thể loại: Film-Noir - Điểm trung bình: 4.36
Thể loại: Action - Điểm trung bình: 3.71
Thể loại: Adventure - Điểm trung bình: 3.63
Thể loại: Sci-Fi - Điểm trung bình: 3.73
Thể loại: Mystery - Điểm trung bình: 4.00
Thể loại: Fantasy - Điểm trung bình: 3.86
Thể loại: Drama - Điểm trung bình: 3.76
Thể loại: Crime - Điểm trung bình: 3.81
Thể loại: Thriller - Điểm trung bình: 3.70
Thể loại: Horror - Điểm trung bình: 4.00


## Bài 3: Phân tích đánh giá theo giới tính

In [4]:
users_rdd = sc.textFile(BASE_PATH + "users.txt").map(lambda line: line.split(","))
user_gender_rdd = users_rdd.map(lambda x: (x[0], x[1]))

user_rating_b3_rdd = ratings_rdd.map(lambda x: (x[0], (x[1], float(x[2]))))
joined_b3_rdd = user_rating_b3_rdd.join(user_gender_rdd)

movie_gender_rating_rdd = joined_b3_rdd.map(lambda x: ((x[1][0][0], x[1][1]), (x[1][0][1], 1)))
movie_gender_total_rdd = movie_gender_rating_rdd.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
movie_gender_avg_rdd = movie_gender_total_rdd.mapValues(lambda x: x[0] / x[1])

print("Điểm trung bình của mỗi phim theo giới tính (10 kết quả đầu):")
for key, avg_rating in movie_gender_avg_rdd.take(10):
    print(f"Phim ID: {key[0]}, Giới tính: {key[1]}, Điểm trung bình: {avg_rating:.2f}")

Điểm trung bình của mỗi phim theo giới tính (10 kết quả đầu):


[Stage 20:======================================>                   (2 + 1) / 3]

Phim ID: 1015, Giới tính: M, Điểm trung bình: 4.33
Phim ID: 1040, Giới tính: M, Điểm trung bình: 3.59
Phim ID: 1010, Giới tính: M, Điểm trung bình: 3.55
Phim ID: 1039, Giới tính: F, Điểm trung bình: 3.90
Phim ID: 1013, Giới tính: F, Điểm trung bình: 3.94
Phim ID: 1037, Giới tính: M, Điểm trung bình: 4.00
Phim ID: 1013, Giới tính: M, Điểm trung bình: 4.06
Phim ID: 1039, Giới tính: M, Điểm trung bình: 3.75
Phim ID: 1050, Giới tính: M, Điểm trung bình: 4.00
Phim ID: 1028, Giới tính: M, Điểm trung bình: 3.50


## Bài 4: Phân tích đánh giá theo nhóm tuổi

In [5]:
def get_age_group(age_str):
    age = int(age_str)
    if age < 18: return "<18"
    elif age <= 35: return "18-35"
    elif age <= 50: return "36-50"
    else: return ">50"

user_age_group_rdd = users_rdd.map(lambda x: (x[0], get_age_group(x[2])))
joined_b4_rdd = user_rating_b3_rdd.join(user_age_group_rdd)

movie_agegroup_rating_rdd = joined_b4_rdd.map(lambda x: ((x[1][0][0], x[1][1]), (x[1][0][1], 1)))
movie_agegroup_total_rdd = movie_agegroup_rating_rdd.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
movie_agegroup_avg_rdd = movie_agegroup_total_rdd.mapValues(lambda x: x[0] / x[1])

print("Điểm trung bình của mỗi phim theo nhóm tuổi (10 kết quả đầu):")
for key, avg_rating in movie_agegroup_avg_rdd.take(10):
    print(f"Phim ID: {key[0]}, Nhóm tuổi: {key[1]}, Điểm trung bình: {avg_rating:.2f}")

Điểm trung bình của mỗi phim theo nhóm tuổi (10 kết quả đầu):
Phim ID: 1010, Nhóm tuổi: >50, Điểm trung bình: 4.50
Phim ID: 1020, Nhóm tuổi: 18-35, Điểm trung bình: 3.56
Phim ID: 1020, Nhóm tuổi: 36-50, Điểm trung bình: 3.83
Phim ID: 1043, Nhóm tuổi: 36-50, Điểm trung bình: 3.94
Phim ID: 1010, Nhóm tuổi: 36-50, Điểm trung bình: 3.29
Phim ID: 1050, Nhóm tuổi: 36-50, Điểm trung bình: 3.64
Phim ID: 1015, Nhóm tuổi: 36-50, Điểm trung bình: 4.50
Phim ID: 1012, Nhóm tuổi: 36-50, Điểm trung bình: 3.50
Phim ID: 1025, Nhóm tuổi: >50, Điểm trung bình: 3.75
Phim ID: 1028, Nhóm tuổi: 36-50, Điểm trung bình: 3.50


## Bài 5: Phân tích đánh giá theo Nghề nghiệp

In [6]:
occ_rdd = sc.textFile(BASE_PATH + "occupation.txt").map(lambda line: line.split(","))
occ_map = occ_rdd.map(lambda x: (x[0], x[1])).collectAsMap()

user_occ_rdd = users_rdd.map(lambda x: (x[0], occ_map.get(x[3], "Unknown")))
user_rating_b5_rdd = ratings_rdd.map(lambda x: (x[0], float(x[2])))

joined_b5_rdd = user_rating_b5_rdd.join(user_occ_rdd)
occ_rating_rdd = joined_b5_rdd.map(lambda x: (x[1][1], (x[1][0], 1)))

occ_total_rdd = occ_rating_rdd.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
occ_avg_rdd = occ_total_rdd.mapValues(lambda x: (x[0] / x[1], x[1]))

print("Trung bình rating và tổng số lượt đánh giá cho từng nghề nghiệp (10 kết quả đầu):")
for occ, (avg_rating, total_count) in occ_avg_rdd.take(10):
    print(f"Nghề: {occ} - Điểm TB: {avg_rating:.2f} - Lượt đánh giá: {total_count}")

Trung bình rating và tổng số lượt đánh giá cho từng nghề nghiệp (10 kết quả đầu):
Nghề: Salesperson - Điểm TB: 3.65 - Lượt đánh giá: 17
Nghề: Doctor - Điểm TB: 3.69 - Lượt đánh giá: 21
Nghề: Accountant - Điểm TB: 3.58 - Lượt đánh giá: 6
Nghề: Designer - Điểm TB: 4.00 - Lượt đánh giá: 13
Nghề: Journalist - Điểm TB: 3.85 - Lượt đánh giá: 17
Nghề: Consultant - Điểm TB: 3.86 - Lượt đánh giá: 14
Nghề: Student - Điểm TB: 4.00 - Lượt đánh giá: 8
Nghề: Manager - Điểm TB: 3.47 - Lượt đánh giá: 16
Nghề: Lawyer - Điểm TB: 3.65 - Lượt đánh giá: 17
Nghề: Nurse - Điểm TB: 3.86 - Lượt đánh giá: 11


## Bài 6: Phân tích đánh giá theo Thời gian (Năm)

In [7]:
def get_year(timestamp_str):
    try:
        ts = int(timestamp_str)
        return datetime.datetime.fromtimestamp(ts).year
    except:
        return "Unknown"

year_rating_rdd = ratings_rdd.map(lambda x: (get_year(x[3]), (float(x[2]), 1)))
year_rating_rdd = year_rating_rdd.filter(lambda x: x[0] != "Unknown")

year_total_rdd = year_rating_rdd.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
year_avg_rdd = year_total_rdd.mapValues(lambda x: (x[0] / x[1], x[1]))
sorted_year_avg_rdd = year_avg_rdd.sortByKey()

print("Tổng số lượt đánh giá và điểm trung bình cho mỗi năm:")
for year, (avg_rating, total_count) in sorted_year_avg_rdd.collect():
    print(f"Năm: {year} - Điểm TB: {avg_rating:.2f} - Tổng lượt: {total_count}")

Tổng số lượt đánh giá và điểm trung bình cho mỗi năm:
Năm: 2020 - Điểm TB: 3.75 - Tổng lượt: 184
